In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import warnings
warnings.filterwarnings("ignore")

print("✓ Done")

✓ Done


In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings; warnings.filterwarnings("ignore")
%matplotlib inline
plt.rcParams['figure.figsize'] = (10, 4)
print("✓ All imports ready")

# ─── LOAD ALL 4 DATASETS ───────────────────────────────────────────
df   = pd.read_csv("Raahi_Behavioral_Chakachakit.csv")
em   = pd.read_csv("vehicle_emission_dataset.csv")
bus  = pd.read_csv("final_bus_data.csv")
fare = pd.read_csv("Raahi_Full_Western_Line_Fares.csv")

print(f"Raahi behavioral : {df.shape}")
print(f"Vehicle emissions: {em.shape}")
print(f"Bus depots       : {bus.shape}")
print(f"WR Fares         : {fare.shape}")

display(df.head(3))
display(df.describe())
display(df.isnull().sum())

# ─── EDA — BEHAVIORAL ─────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

df['crowd_grade'].value_counts().plot(kind='bar', ax=axes[0],
    color=['#1D9E75','#EF9F27','#E24B4A'])
axes[0].set_title('Crowd grade distribution')

df.groupby('hour')['passenger_count'].mean().plot(kind='line', ax=axes[1],
    color='#534AB7', marker='o', markersize=4)
axes[1].set_title('Avg passenger count by hour')

axes[2].scatter(df['congestion_perc'], df['aqi_value'],
    alpha=0.3, color='#D85A30', s=15)
axes[2].set_xlabel('Congestion %'); axes[2].set_ylabel('AQI')
axes[2].set_title('AQI vs road congestion')

plt.tight_layout(); plt.show()

# ─── EDA — FARES ──────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

fare['2nd_Class_Fare'].hist(bins=20, ax=axes[0], color='#1D9E75', alpha=0.8)
axes[0].set_title('2nd Class fare distribution')
axes[0].set_xlabel('Fare (₹)')

axes[1].scatter(fare['Distance_KM'], fare['2nd_Class_Fare'],
    alpha=0.4, color='#534AB7', s=15, label='2nd Class')
axes[1].scatter(fare['Distance_KM'], fare['1st_Class_Fare'],
    alpha=0.4, color='#EF9F27', s=15, label='1st Class')
axes[1].scatter(fare['Distance_KM'], fare['AC_Local_Fare'],
    alpha=0.4, color='#E24B4A', s=15, label='AC Local')
axes[1].set_title('Distance vs Fare by class')
axes[1].set_xlabel('Distance (km)'); axes[1].set_ylabel('Fare (₹)')
axes[1].legend()

plt.tight_layout(); plt.show()

# ─── EDA — EMISSIONS ──────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

em.groupby('Vehicle Type')['CO2 Emissions'].mean().sort_values().plot(
    kind='barh', ax=axes[0], color=['#1D9E75','#EF9F27','#E24B4A','#534AB7'])
axes[0].set_title('Avg CO₂ by vehicle type')

em['Emission Level'].value_counts().plot(kind='pie', ax=axes[1],
    colors=['#1D9E75','#EF9F27','#E24B4A'],
    autopct='%1.1f%%', startangle=90)
axes[1].set_title('Emission level distribution'); axes[1].set_ylabel('')

plt.tight_layout(); plt.show()

# ─── EDA — BUS DEPOTS ─────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

bus['sub_type'].value_counts().plot(kind='bar', ax=axes[0],
    color=['#534AB7','#1D9E75'])
axes[0].set_title('Bus depot sub-types'); axes[0].tick_params(rotation=0)

axes[1].scatter(bus['geo_long'], bus['geo_lat'],
    color='#EF9F27', alpha=0.7, s=40, edgecolors='white', linewidths=0.5)
axes[1].set_title('Bus depot locations (Mumbai)')
axes[1].set_xlabel('Longitude'); axes[1].set_ylabel('Latitude')

plt.tight_layout(); plt.show()

# ══════════════════════════════════════════════════════════════════
# MODEL 1 — ADVISORY CLASSIFIER
# Source: Raahi_Behavioral_Chakachakit + vehicle_emission_dataset
# ══════════════════════════════════════════════════════════════════

df['log_passenger_count'] = np.log1p(df['passenger_count'])

def peak_weight(hour, is_peak):
    if is_peak == 1:
        return 3 if hour in [8,9,18,19] else 2 if hour in [7,10,17,20] else 1.5
    return 1.0
df['peak_hour_weight'] = df.apply(lambda r: peak_weight(r['hour'], r['is_peak']), axis=1)

def monsoon_score(r):
    return round(min(0.5*r['is_raining'] + r['congestion_perc']/200 + r['predicted_delay']/30*0.2, 1.0), 4)
df['monsoon_impact'] = df.apply(monsoon_score, axis=1)

df['aqi_category']  = pd.cut(df['aqi_value'], bins=[0,50,100,200,300,400,999],
                               labels=[0,1,2,3,4,5]).astype(int)
df['comfort_score'] = (((1-df['crowd_grade']/2)*.5) + ((1-df['is_raining'])*.2) +
                        ((1-df['aqi_category']/5)*.3)).round(4)
df['speed_diff']    = df['avg_road_speed'] - (60 - df['predicted_delay'])

flood_stations      = {"DADAR","MAHIM JN","BANDRA","ANDHERI","JOGESHWARI"}
df['flood_prone']   = df['station_name'].isin(flood_stations).astype(int)
df['flood_risk']    = df['flood_prone'] * df['is_raining']

pm25     = em[em['Vehicle Type']=='Bus'].groupby('Traffic Conditions')['PM2.5 Emissions'].mean()
tc_map   = lambda p: 'Free flow' if p<40 else ('Heavy' if p>80 else 'Moderate')
df['pm25_emission_proxy'] = df['congestion_perc'].apply(tc_map).map(pm25)

print(f"✓ Features added. Shape: {df.shape}")
display(df[['comfort_score','monsoon_impact','flood_risk','aqi_category']].describe())

def make_advisory(r):
    sdcl   = r['crowd_grade'] == 2
    dense  = r['crowd_grade'] == 1
    rain   = r['is_raining']  == 1
    flood  = r['flood_risk']  == 1
    badaqi = r['aqi_category'] >= 3
    roadok = r['congestion_perc'] < 60
    fast   = r['avg_road_speed'] > 35
    delay  = r['predicted_delay'] > 15
    if (sdcl and rain) or (flood and badaqi): return "avoid_delay"
    if sdcl and roadok and not flood:          return "switch_bus"
    if (sdcl or dense) and fast and not badaqi: return "private_advised"
    return "train_ok"

df['advisory'] = df.apply(make_advisory, axis=1)

df['advisory'].value_counts().plot(kind='barh',
    color=['#1D9E75','#EF9F27','#378ADD','#E24B4A'])
plt.title('Advisory label distribution'); plt.show()

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report

FEATURES = ['hour','is_peak','peak_hour_weight','log_passenger_count',
            'crowd_grade','congestion_perc','avg_road_speed','speed_diff',
            'aqi_value','aqi_category','is_raining','monsoon_impact',
            'flood_risk','comfort_score','predicted_delay','pm25_emission_proxy']

X  = df[FEATURES]
le = LabelEncoder()
y  = le.fit_transform(df['advisory'])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42)

clf = RandomForestClassifier(n_estimators=200, max_depth=12,
      class_weight='balanced', random_state=42, n_jobs=-1)
clf.fit(X_train, y_train)

cv = cross_val_score(clf, X, y, cv=5)
print(f"CV accuracy: {cv.mean()*100:.2f}% ± {cv.std()*100:.2f}%")
print(classification_report(y_test, clf.predict(X_test), target_names=le.classes_))

imp = pd.Series(clf.feature_importances_, index=FEATURES).sort_values()
colors = ['#534AB7' if v > 0.10 else '#9FE1CB' for v in imp]
imp.plot(kind='barh', color=colors, figsize=(8,5))
plt.title('Feature importance — Advisory model')
plt.xlabel('Importance score'); plt.tight_layout(); plt.show()

imp.sort_values(ascending=False).to_csv('raahi_feature_importance.csv')

# ══════════════════════════════════════════════════════════════════
# MODEL 2 — FARE PREDICTOR
# Source: Raahi_Full_Western_Line_Fares.csv
# ══════════════════════════════════════════════════════════════════

from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, r2_score

fare_df = fare.copy()

le_from = LabelEncoder()
le_to   = LabelEncoder()
fare_df['from_enc']       = le_from.fit_transform(fare_df['From'])
fare_df['to_enc']         = le_to.fit_transform(fare_df['To'])
fare_df['distance_sq']    = fare_df['Distance_KM'] ** 2
fare_df['distance_bucket']= pd.cut(fare_df['Distance_KM'],
    bins=[0,10,25,50,80,200], labels=[0,1,2,3,4]).astype(int)

FARE_FEATURES = ['Distance_KM','distance_sq','distance_bucket','from_enc','to_enc']

fare_results = {}
for target in ['2nd_Class_Fare','1st_Class_Fare','AC_Local_Fare']:
    Xf = fare_df[FARE_FEATURES]
    yf = fare_df[target]
    Xf_tr, Xf_te, yf_tr, yf_te = train_test_split(Xf, yf, test_size=0.2, random_state=42)
    fm = GradientBoostingRegressor(n_estimators=200, max_depth=4,
         learning_rate=0.05, random_state=42)
    fm.fit(Xf_tr, yf_tr)
    preds = fm.predict(Xf_te)
    fare_results[target] = {'model': fm,
                             'mae': mean_absolute_error(yf_te, preds),
                             'r2':  r2_score(yf_te, preds)}
    print(f"  {target:<20} MAE: ₹{fare_results[target]['mae']:.2f}   R²: {fare_results[target]['r2']:.4f}")

print("✓ Fare models trained")

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, target in zip(axes, ['2nd_Class_Fare','1st_Class_Fare','AC_Local_Fare']):
    Xf = fare_df[FARE_FEATURES]; yf = fare_df[target]
    _, Xf_te, _, yf_te = train_test_split(Xf, yf, test_size=0.2, random_state=42)
    preds = fare_results[target]['model'].predict(Xf_te)
    ax.scatter(yf_te, preds, alpha=0.5, color='#534AB7', s=20)
    ax.plot([yf_te.min(), yf_te.max()], [yf_te.min(), yf_te.max()],
            color='#E24B4A', linewidth=1.5, linestyle='--')
    ax.set_title(f"{target}\nMAE:₹{fare_results[target]['mae']:.2f}  R²:{fare_results[target]['r2']:.3f}")
    ax.set_xlabel('Actual (₹)'); ax.set_ylabel('Predicted (₹)')
plt.tight_layout(); plt.show()

# ══════════════════════════════════════════════════════════════════
# MODEL 3 — EMISSION LEVEL CLASSIFIER
# Source: vehicle_emission_dataset.csv
# ══════════════════════════════════════════════════════════════════

em_df = em.copy()

le_vtype    = LabelEncoder()
le_fuel     = LabelEncoder()
le_road     = LabelEncoder()
le_traffic  = LabelEncoder()
le_emlevel  = LabelEncoder()

em_df['vehicle_type_enc'] = le_vtype.fit_transform(em_df['Vehicle Type'])
em_df['fuel_type_enc']    = le_fuel.fit_transform(em_df['Fuel Type'])
em_df['road_type_enc']    = le_road.fit_transform(em_df['Road Type'])
em_df['traffic_cond_enc'] = le_traffic.fit_transform(em_df['Traffic Conditions'])

EM_FEATURES = [
    'vehicle_type_enc','fuel_type_enc','Engine Size','Age of Vehicle',
    'Mileage','Speed','Acceleration','road_type_enc','traffic_cond_enc',
    'Temperature','Humidity','Wind Speed','Air Pressure',
    'CO2 Emissions','NOx Emissions','PM2.5 Emissions'
]

Xe = em_df[EM_FEATURES]
ye = le_emlevel.fit_transform(em_df['Emission Level'])

Xe_tr, Xe_te, ye_tr, ye_te = train_test_split(
    Xe, ye, test_size=0.2, stratify=ye, random_state=42)

clf_em = RandomForestClassifier(n_estimators=150, max_depth=10,
         class_weight='balanced', random_state=42, n_jobs=-1)
clf_em.fit(Xe_tr, ye_tr)

cv_em = cross_val_score(clf_em, Xe, ye, cv=5)
print(f"CV accuracy: {cv_em.mean()*100:.2f}% ± {cv_em.std()*100:.2f}%")
print(classification_report(ye_te, clf_em.predict(Xe_te), target_names=le_emlevel.classes_))

imp_em = pd.Series(clf_em.feature_importances_, index=EM_FEATURES).sort_values()
imp_em.plot(kind='barh', color='#534AB7', figsize=(10,5))
plt.title('Feature importance — Emission model')
plt.xlabel('Importance score'); plt.tight_layout(); plt.show()

# ══════════════════════════════════════════════════════════════════
# BUS DEPOT MAP LAYER
# Source: final_bus_data.csv — no model, used as Folium map layer
# ══════════════════════════════════════════════════════════════════

bus_clean = bus[['name','geo_lat','geo_long','sub_type','ward']].dropna()
print(f"✓ Bus depots ready for map: {len(bus_clean)} locations")
display(bus_clean.head(5))

# ─── SAVE ALL MODELS ──────────────────────────────────────────────
import joblib

# Model 1 — advisory
joblib.dump(clf,    'model.pkl')
joblib.dump(le,     'label_encoder.pkl')

# Model 2 — fare
joblib.dump(fare_results['2nd_Class_Fare']['model'], 'fare_model_2nd.pkl')
joblib.dump(fare_results['1st_Class_Fare']['model'], 'fare_model_1st.pkl')
joblib.dump(fare_results['AC_Local_Fare']['model'],  'fare_model_ac.pkl')
joblib.dump(le_from, 'fare_encoder_from.pkl')
joblib.dump(le_to,   'fare_encoder_to.pkl')
fare.to_csv('fare_lookup.csv', index=False)

# Model 3 — emission
joblib.dump(clf_em,     'emission_model.pkl')
joblib.dump(le_emlevel, 'emission_label_encoder.pkl')
joblib.dump(le_vtype,   'emission_enc_vehicle.pkl')
joblib.dump(le_fuel,    'emission_enc_fuel.pkl')
joblib.dump(le_road,    'emission_enc_road.pkl')
joblib.dump(le_traffic, 'emission_enc_traffic.pkl')

# Bus depot clean export
bus_clean.to_csv('bus_depots.csv', index=False)

# Full dataset with predictions
df.to_csv('raahi_with_predictions.csv', index=False)

print("✓ model.pkl saved")
print("✓ label_encoder.pkl saved")
print("✓ fare_model_2nd/1st/ac.pkl saved")
print("✓ emission_model.pkl saved")
print("✓ bus_depots.csv saved")
print("✓ raahi_with_predictions.csv saved")
print("→ Copy ALL .pkl and .csv files into your Flask project folder")

# ─── LIVE TEST WIDGET ─────────────────────────────────────────────
import ipywidgets as widgets
from IPython.display import display, clear_output

station_w = widgets.Dropdown(options=df['station_name'].unique(), description='Station')
hour_w    = widgets.IntSlider(min=0, max=23, value=8, description='Hour')
crowd_w   = widgets.RadioButtons(options=[('Normal',0),('Dense',1),('SDCL',2)], description='Crowd')
rain_w    = widgets.Checkbox(value=False, description='Raining')
cong_w    = widgets.IntSlider(min=15, max=129, value=45, description='Congestion')
aqi_w     = widgets.IntSlider(min=50, max=400, value=185, description='AQI')
out       = widgets.Output()

def on_change(_):
    with out:
        clear_output(wait=True)
        row = df[(df['station_name']==station_w.value)&(df['hour']==hour_w.value)]
        row = row.iloc[0].copy() if len(row) else df.iloc[0].copy()
        row['crowd_grade']     = crowd_w.value
        row['is_raining']      = int(rain_w.value)
        row['congestion_perc'] = cong_w.value
        row['aqi_value']       = aqi_w.value
        row['aqi_category']    = min(5, aqi_w.value//80)
        row['comfort_score']   = round((1-crowd_w.value/2)*.5+(1-int(rain_w.value))*.2, 3)
        row['flood_risk']      = int(rain_w.value) if station_w.value in flood_stations else 0
        row['monsoon_impact']  = round(min(0.5*int(rain_w.value)+cong_w.value/200, 1.0), 4)
        pred  = clf.predict([row[FEATURES]])[0]
        conf  = clf.predict_proba([row[FEATURES]]).max()
        label = le.inverse_transform([pred])[0]
        emoji = {'train_ok':'🟢','switch_bus':'🟡','private_advised':'🔵','avoid_delay':'🔴'}
        print(f"{emoji[label]}  Advisory: {label.upper()}  ({conf*100:.0f}% confidence)")

        # Show fare from this station to Churchgate
        station_fare = fare[fare['From']==station_w.value.title()]
        if len(station_fare) > 0:
            row_f = station_fare.iloc[0]
            print(f"\n   To {row_f['To']} ({row_f['Distance_KM']} km)")
            print(f"   2nd ₹{row_f['2nd_Class_Fare']} | 1st ₹{row_f['1st_Class_Fare']} | AC ₹{row_f['AC_Local_Fare']}")

for w in [station_w, hour_w, crowd_w, rain_w, cong_w, aqi_w]:
    w.observe(on_change, names='value')

display(widgets.VBox([station_w, hour_w, crowd_w, rain_w, cong_w, aqi_w, out]))
on_change(None)

🟢  Advisory: TRAIN_OK  (97% confidence)
